In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
===========================================================
Vehicle Data Retrieval Tool (Tkinter GUI + API Integration)
===========================================================

Author   : Sanmathi S
Year     : 2025
Version  : 1.0
Purpose  :
    This tool provides a Tkinter-based GUI to fetch vehicle data
    from the Darby API using Asset IDs and VINs. It allows users
    to select message types and date ranges, then downloads the
    results into CSV files sorted by event time.

Features :
    - Upload Asset ID/VIN list from CSV
    - Select message type from predefined options
    - Choose start and end dates via calendar widget
    - Fetch data from API with pagination
    - Save results to CSV (one file per VIN)
    - Progress updates during download
    - Clear/reset fields with a single click

Output   :
    - CSV files saved in user-selected folder
    - Each file named <VIN>.csv containing API results
    - GUI notifications for success or errors

Dependencies:
    - requests
    - tkinter
    - tkcalendar
    - pandas
    - csv
    - os
    - datetime
    - threading



API Endpoint:
POST Your_API

Request Body:
{
    "filter": {
        "startTime": "YYYY-MM-DDTHH:MM:SSZ",
        "endTime": "YYYY-MM-DDTHH:MM:SSZ",
        "messageType": "..."
    },
    "page": 0,
    "limit": 3000
}

Response Format:
{
    "results": [ { message payload... } ]
}



Usage    :
    python vehicle_data_retrieval.py

Notes:
- Ensure the bearer token is valid before running.
- Input CSV must contain columns: "Asset Id" and "VIN".
- Output files are saved with VIN-based filenames in the selected folder.

"""





import requests
import tkinter as tk
from tkinter import messagebox, filedialog
from tkcalendar import DateEntry
from tkinter.ttk import Combobox
from datetime import datetime, timedelta
import csv
import os
 
# -------------------
# Config
# -------------------
API_BASE = "Your_API"
BEARER_TOKEN = "Your_Token"  # Replace with your valid token
loaded_asset_ids = []  # List of dicts: {"asset_id": ..., "vin": ...}
 
# -------------------
# Fetch API Data
# -------------------
def fetch_messages(asset_id, message_type, start_time, end_time):
    headers = {"Authorization": f"Bearer {BEARER_TOKEN}"}
    all_messages = []
    page = 0
    limit = 3000
 
    while True:
        params = {
            "filter": {
                "startTime": start_time,
                "endTime": end_time,
                "messageType": message_type
            },
            "page": page,
            "limit": limit
        }
 
        url = f"{API_BASE}{asset_id}/search"
        response = requests.post(url, headers=headers, json=params)
 
        if response.status_code != 200:
            raise Exception(f"API Error: {response.status_code}, {response.text}")
 
        data = response.json()
        messages = data.get("results", []) if isinstance(data, dict) else data
        if not messages:
            break
 
        all_messages.extend(messages)
        if len(messages) < limit:
            break
        page += 1
 
    return {"results": all_messages}
 
# -------------------
# Save to CSV using VIN
# -------------------


# with asending (eventtime)

def save_to_csv_auto(data, folder_path, vin):
    payload = data.get("results", [])
    if not payload:
        print(f"No messages found for VIN: {vin}")
        return

    flat_rows = [item for item in payload if isinstance(item, dict)]
    if not flat_rows:
        print(f"No valid rows for VIN: {vin}")
        return

    # Add VIN to each row
    for row in flat_rows:
        row["Vin"] = vin

    # Sort rows by eventTime (ascending)
    flat_rows.sort(key=lambda x: x.get("eventTime", ""))

    file_path = os.path.join(folder_path, f"{vin}.csv")
    all_keys = set()
    for row in flat_rows:
        all_keys.update(row.keys())
    fieldnames = sorted(all_keys)

    with open(file_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in flat_rows:
            writer.writerow(row)

    print(f"✅ Saved: {file_path}")


def load_asset_ids_from_csv():
    global uploaded_file_name
    file_path = filedialog.askopenfilename(filetypes=[("CSV Files", "*.csv")])
    if file_path:
        uploaded_file_name = os.path.basename(file_path)
        input_file_label.config(text=f"Input File: {uploaded_file_name}")
        loaded_asset_ids.clear()
        with open(file_path, "r") as f:
            reader = csv.DictReader(f)
            for row in reader:
                asset_id = row.get("Asset Id", "").strip()
                vin = row.get("VIN", "").strip()
                if asset_id and vin:
                    loaded_asset_ids.append({"asset_id": asset_id, "vin": vin})
        messagebox.showinfo("Loaded", f"{len(loaded_asset_ids)} Asset IDs loaded.")


 
# -------------------
# Download Button Action
# -------------------
def download_data():
    msg_type = msg_type_combo.get().strip()
    start_date = start_date_entry.get_date()
    end_date = end_date_entry.get_date()
 
    if not msg_type:
        messagebox.showerror("Validation Error", "Message Type is required.")
        return
    if end_date < start_date:
        messagebox.showerror("Date Error", "End Date must be after Start Date.")
        return
 
    start_time = datetime.combine(start_date, datetime.min.time()).strftime("%Y-%m-%dT%H:%M:%SZ")
    end_time = datetime.combine(end_date, datetime.max.time()).strftime("%Y-%m-%dT%H:%M:%SZ")
    folder_path = filedialog.askdirectory(title="Select Folder to Save CSV Files")
    if not folder_path:
        messagebox.showerror("Cancelled", "Folder selection cancelled.")
        return
 
    if not loaded_asset_ids:
        messagebox.showerror("Validation Error", "No Asset IDs loaded.")
        return
 
    saved_count = 0
    total_assets = len(loaded_asset_ids)

        
        
    for index, entry in enumerate(loaded_asset_ids, start=1):
        asset_id = entry["asset_id"]
        vin = entry["vin"]
        
        
        import time  # Add this at the top

        # Inside your loop, after updating the label:
        progress_label.config(text=f"Processing VIN: {vin} ({index} of {total_assets})")
        root.update_idletasks()
        time.sleep(0.3)  # Add a small delay (300ms)

        
        try:
            data = fetch_messages(asset_id, msg_type, start_time, end_time)
            if data.get("results"):
                save_to_csv_auto(data, folder_path, vin)
                saved_count += 1
        except Exception as e:
            print(f"Error fetching data for {asset_id} ({vin}): {e}")


    
    messagebox.showinfo("Completed", f"{saved_count} files saved to:\n{folder_path}")
    progress_label.config(text="")
 

def clear_fields():
    global uploaded_file_name
    msg_type_combo.set("")
    start_date_entry.set_date(seven_days_ago)
    end_date_entry.set_date(today)
    loaded_asset_ids.clear()
    uploaded_file_name = ""
    input_file_label.config(text="Input File: None")
    progress_label.config(text="")
    messagebox.showinfo("Cleared", "All fields cleared.")

# -------------------
# UI Setup
# -------------------
PINK = "#E5E6F0"
root = tk.Tk()
root.title("Vehicle Data Retrieval Tool")
root.geometry("800x600")
root.configure(bg=PINK)
 
label_font = ('calibri', 13, 'bold')
 
tk.Label(root, text="Vehicle Data Retrieval Interface", font=('Arial Black', 15, 'bold'), fg="#353E9E", bg=PINK).pack(pady=(30, 10))
 
# Top Frame
top_frame = tk.Frame(root, bg=PINK)
top_frame.pack(pady=(20, 10))
 
upload_btn = tk.Button(top_frame, text="Upload Asset ID CSV", command=load_asset_ids_from_csv,
                       bg="#8A6ED6", fg="white", font=('Helvetica', 12, 'bold'))
upload_btn.pack()

# input_file_label = tk.Label(top_frame, text="Input File: None", font=label_font, bg=PINK, fg="#42488A")
# input_file_label.pack(pady=5)

input_file_label = tk.Label(
    top_frame,
    text="Input File: None",
    font=('calibri', 12, 'bold'),
    bg="#D6D6F0",       # Slightly different shade for contrast
    fg="#2E3A87",       # Deeper blue for better readability
    relief="groove",    # Adds a border effect
    bd=2,               # Border width
    padx=10,            # Horizontal padding inside label
    pady=5              # Vertical padding inside label
)
input_file_label.pack(pady=(10, 5))  # External padding around the label



# Middle Frame
middle_frame = tk.Frame(root, bg=PINK)
middle_frame.pack(pady=10)
 
tk.Label(middle_frame, text="Select Message Type:", font=label_font, bg=PINK).pack(pady=5)
# msg_type_options = ["EV_169_PERIODIC", "ALPD_OBD_EVENT", "ALPD_GPS_DATA", "EDC_BS6_PERIODIC" ]


msg_type_options = ["EV_169_PERIODIC", "EDC_BS6_PERIODIC", "EDC_BS6_CNG_PERIODIC", "LCV_BS6_ZD30_PERIODIC", "LCV_BS6_OBD2_PERIODIC" ]

msg_type_combo = Combobox(middle_frame, values=msg_type_options, width=37, state="readonly")
msg_type_combo.pack()
 
today = datetime.today()
seven_days_ago = today - timedelta(days=6)
 
tk.Label(middle_frame, text="Start Date:", font=label_font, bg=PINK).pack(pady=5)
start_date_entry = DateEntry(middle_frame, width=38, mindate=seven_days_ago, maxdate=today, date_pattern="yyyy-mm-dd")
start_date_entry.set_date(seven_days_ago)
start_date_entry.pack()
 
tk.Label(middle_frame, text="End Date:", font=label_font, bg=PINK).pack(pady=5)
end_date_entry = DateEntry(middle_frame, width=38, mindate=seven_days_ago, maxdate=today, date_pattern="yyyy-mm-dd")
end_date_entry.set_date(today)
end_date_entry.pack()
 
# Bottom Frame
bottom_frame = tk.Frame(root, bg=PINK)
bottom_frame.pack(side="bottom", pady=20)
 
tk.Button(bottom_frame, text="Download Data", command=download_data,
          bg="#007204", fg="white", font=('Helvetica', 12, 'bold')).pack(pady=10)
 
tk.Button(bottom_frame, text="Clear", command=clear_fields,
          bg="#E73737", fg="white", font=('Helvetica', 12, 'bold')).pack(pady=5)
 
progress_label = tk.Label(bottom_frame, text="", font=label_font, bg=PINK, fg="#42488A")
progress_label.pack(pady=5)
 
root.mainloop()

